In [11]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Load model & tokenizer
model_name = "./models/big-qwen"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

# 2. Prepare input
input_text2 = """
You are a game command parser that converts natural language commands into DSL instructions.
Game State:
AIMED _ AT:
  type: Wall
  distance: 330.86
  interactable: yes

MONSTERS (count=0):

INVENTORY:
  current _ slot: 2
  weapons:
    - (1, Fist, 0)
    - (2, Pistol, 50)
Command:
Go press that switch ahead
"""
output_reference2 = "INTERACT"

input_text = """
You are a game command parser that converts natural language commands into DSL instructions.
Game State:
AIMED_AT:
  type: Wall
  distance: 467.27
  interactable: yes

MONSTERS (count=2):
  - (MONSTER_0, Zombieman, 20, 228.43, 29.20, 3.37)
  - (MONSTER_1, Zombieman, 20, 218.13, 11.99, 4.33)

INVENTORY:
  current_slot: 3
  weapons:
    - (1, Fist, 0)
    - (2, Pistol, 186)
    - (3, Shotgun, 50)
Command:
Ignore the switch and gun down those soldiers
"""
output_reference = "MONSTER_0"

inputs = tokenizer(input_text, return_tensors="pt")
input_ids = inputs["input_ids"]

# Storage for the gradient
embedding_grads = []

def backward_hook(grad):
    embedding_grads.append(grad)

# Get embeddings and register hook BEFORE forward pass
embedding_layer = model.get_input_embeddings()
embeddings = embedding_layer(input_ids)
embeddings.requires_grad_(True)
embeddings.register_hook(backward_hook)

target_token_id = tokenizer(output_reference, add_special_tokens=False)["input_ids"][-1]
print(tokenizer(output_reference, add_special_tokens=False)["input_ids"])

outputs = model(inputs_embeds=embeddings)
logits = outputs.logits

last_logits = logits[0, -1, :]
log_probs = torch.nn.functional.log_softmax(last_logits, dim=-1)
loss = -log_probs[target_token_id]

loss.backward()

# Use the captured gradient
saliency = embedding_grads[0].norm(dim=-1).squeeze(0)

tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
tokens_map = list()
for token, score in zip(tokens, saliency):
    print(f"{token}: {score.item():.4f}")
    tokens_map.append((token, score.item()))

json.dump(tokens_map, open("test.json", "w"), indent=2)

[21344, 37923, 62, 15]
Ċ: 4.7877
You: 1.1788
Ġare: 0.6861
Ġa: 4.4880
Ġgame: 1.7163
Ġcommand: 1.7393
Ġparser: 0.8495
Ġthat: 0.7563
Ġconverts: 1.7905
Ġnatural: 1.0077
Ġlanguage: 0.7796
Ġcommands: 1.2226
Ġinto: 1.6174
ĠDSL: 2.8164
Ġinstructions: 3.0483
.Ċ: 2.7412
Game: 3.5678
ĠState: 2.5371
:Ċ: 0.6994
AIM: 1.9029
ED: 0.5616
_AT: 1.1467
:Ċ: 0.4504
Ġ: 0.1894
Ġtype: 0.3812
:: 0.2321
ĠWall: 0.5337
Ċ: 0.4301
Ġ: 0.4981
Ġdistance: 0.9326
:: 0.6384
Ġ: 1.3964
4: 2.6107
6: 3.8063
7: 1.5778
.: 0.4409
2: 0.5489
7: 0.3443
Ċ: 0.4944
Ġ: 0.2134
Ġinteract: 1.7011
able: 1.1316
:: 0.5465
Ġyes: 1.1620
ĊĊ: 0.5420
MON: 0.3018
ST: 0.3925
ERS: 0.3114
Ġ(: 0.6084
count: 0.8412
=: 1.0260
2: 0.6517
):Ċ: 0.9326
Ġ: 0.1775
Ġ-: 0.4042
Ġ(: 0.1987
MON: 0.2774
STER: 0.3885
_: 0.6394
0: 0.5573
,: 0.6146
ĠZ: 0.8504
omb: 0.3338
i: 0.6046
eman: 1.3629
,: 0.9335
Ġ: 0.7778
2: 1.5978
0: 1.1312
,: 1.2715
Ġ: 1.0531
2: 2.5346
2: 1.8422
8: 1.3507
.: 1.6547
4: 1.4180
3: 1.0984
,: 1.1304
Ġ: 1.3380
2: 2.8287
9: 2.4412
.: 3.6662
2: 2.770